# CTB ProSiT reproduction

This notebook loads the saved CTB Petri net and ProSiT parameter bundles and reproduces the numerical results reported in the thesis.

The confidential terminal event log is not included. The saved workload-aware baseline and the two what-if models are simulated again from the PKL bundles. Historical hold-out, state-ablation, drift, structural-repair, bottleneck, and capacity results are recalculated from the included per-seed or derived evidence. They cannot be regenerated from the confidential raw events.

## 1. Connect Google Drive and create an isolated environment

Upload this complete folder to `MyDrive/reproducibility`. The isolated environment prevents Colab's preinstalled NumPy and pandas versions from being used by the reproduction.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import shutil
import subprocess
import sys
from pathlib import Path

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "This reproduction uses Python 3.11. In Colab select "
        "Runtime > Change runtime type > Runtime Version > 2025.07, "
        "then run the notebook again."
    )

# Change this line only if the uploaded folder has another name or location.
REPRO_DIR = Path("/content/drive/MyDrive/reproducibility")
if not (REPRO_DIR / "reviewer_runner.py").is_file():
    raise FileNotFoundError(f"Reproduction folder not found: {REPRO_DIR}")

os.chdir(REPRO_DIR)

VENV = Path("/content/ctb_prosit_env")
PYTHON = VENV / "bin" / "python"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "virtualenv"],
    check=True,
)

if not PYTHON.is_file():
    subprocess.run(
        [sys.executable, "-m", "virtualenv", str(VENV)],
        check=True,
    )

subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)

ENV = os.environ.copy()
ENV["MPLBACKEND"] = "Agg"
ENV["PYTHONUNBUFFERED"] = "1"

def run_in_reproduction_environment(source):
    subprocess.run(
        [str(PYTHON), "-c", source],
        cwd=REPRO_DIR,
        env=ENV,
        check=True,
    )

print("Reproduction environment:", PYTHON)

## Saved model files

The package follows ProSiT's documented save/load approach:

- PNML stores the Petri-net control flow.
- JSON is the readable parameter export produced by `SimulatorParameters.to_json()`.
- PKL stores the exact calibrated Python object used for the thesis simulations.

The PKL is required for exact reproduction because ProSiT 1.0.3 does not restore the empirical sample arrays stored in the calibrated CTB rule leaves from JSON. The notebook demonstrates the JSON API and reports this difference. Only load the verified PKL files supplied in this folder.

## 2. Verify and load the saved models

In [ ]:
run_in_reproduction_environment('\nfrom pathlib import Path\nimport pickle\n\nimport pandas as pd\nimport pm4py\nfrom prosit import SimulatorParameters, SimulatorEngine\n\nimport reviewer_runner as rr\n\nintegrity = rr.verify_package_files()\nprint(f"Verified frozen files: {len(integrity)}")\nprint(rr.package_versions().to_string(index=False))\n\nnet, initial_marking, final_marking = pm4py.read_pnml(\n    "models/ctb_inductive_miner.pnml"\n)\nprint(\n    f"Petri net: {len(net.places)} places, "\n    f"{len(net.transitions)} transitions, {len(net.arcs)} arcs"\n)\n\nwith open("models/params_baseline_rmg_max_concurrency_3.pkl", "rb") as handle:\n    baseline = pickle.load(handle)\n\nengine = SimulatorEngine(baseline)\nprint(f"Loaded executable baseline: {type(engine).__name__}")\n\nmodels = rr.load_models()\nprint("\\nSaved model configurations")\nprint(rr.model_summary(models).to_string(index=False))\n\nprint("\\nModel contract checks")\nprint(rr.assert_model_contracts(models).to_string(index=False))\n\njson_report = rr.export_and_reload_official_json(\n    baseline, Path("outputs/json_api_demo.json")\n)\nprint("\\nProSiT JSON export/import check")\nprint(pd.Series(json_report).to_string())\n')

## 3. Reconstruct the remaining thesis results

These tables are recomputed from the included ten-seed validation outputs and non-confidential derived evidence.

In [ ]:
run_in_reproduction_environment('\nfrom pathlib import Path\nimport json\n\nimport pandas as pd\nimport reviewer_runner as rr\n\noutput_dir = Path("outputs")\noutput_dir.mkdir(exist_ok=True)\n\nhistorical_summary, historical_contrasts = rr.reconstruct_historical_ablation()\nhistorical_summary.to_csv(output_dir / "historical_ablation_summary.csv", index=False)\nhistorical_contrasts.to_csv(output_dir / "historical_ablation_contrasts.csv", index=False)\n\nprint("Historical three-state ablation")\nheadline = historical_summary[historical_summary["metric"].isin([\n    "case_turnaround_emd_min",\n    "case_turnaround_sim_mean",\n    "case_turnaround_sim_p90",\n    "yard_service_time_emd_frequency_weighted_min",\n    "yard_activity_rate_l1_error",\n    "gate_only_cases",\n])]\nprint(headline.to_string(index=False))\n\nprint("\\nPaired state contrasts")\nprint(historical_contrasts.to_string(index=False))\n\nevidence = rr.load_claim_evidence()\n\ntemporal = evidence["temporal_transfer"]\nprint("\\nTemporal transfer")\nprint(pd.Series({\n    "train_mean_turnaround_min": temporal["turnaround"]["train_mean_min"],\n    "test_mean_turnaround_min": temporal["turnaround"]["test_mean_min"],\n    "mean_shift_min": temporal["turnaround"]["mean_shift"]["difference_test_minus_train"],\n    "mean_shift_ci95_lo": temporal["turnaround"]["mean_shift"]["ci95_lo"],\n    "mean_shift_ci95_hi": temporal["turnaround"]["mean_shift"]["ci95_hi"],\n    "p90_shift_min": temporal["turnaround"]["p90_shift"]["difference_test_minus_train"],\n    "yard_service_weighted_emd_min": temporal["yard_service_frequency_weighted_wasserstein_min"],\n}).to_string())\n\nrepair = evidence["structural_repair"]\nprint("\\nStructural repair")\nrepair_table = pd.DataFrame([\n    {"model": "discovered", **repair["before"]["test"]},\n    {"model": "gate_only_restricted", **repair["after"]["test"]},\n])\nprint(repair_table[[\n    "model", "fitness", "precision", "generalization", "simplicity"\n]].to_string(index=False))\n\nprint("\\nBottleneck ranking")\nprint(evidence["bottleneck_ranking"].head(10).to_string(index=False))\n\ncapacity = evidence["capacity_pressure"]\nprint("\\nRMG capacity pressure")\nprint(pd.Series({\n    "highest_pressure_block": capacity["highest_pressure_block"],\n    "baseline_nominal_utilization": capacity["highest_baseline_nominal_utilization"],\n    "demand_plus_20_nominal_utilization": capacity["highest_scenario_nominal_utilization"],\n    "multiplier_to_mean_saturation": capacity["smallest_multiplier_to_mean_saturation"],\n    "blocks_with_minutes_above_capacity": capacity["blocks_with_observed_minutes_above_capacity"],\n}).to_string())\n\nprint("\\nScenario state-ablation contrasts")\nstate_ablation = evidence["scenario_state_ablation"]\nstate_ablation = state_ablation[state_ablation["metric"].isin([\n    "mean_turnaround_min",\n    "mean_rmg_service_min",\n    "mean_rmg_pre_service_min",\n    "mean_rmg_receive_service_min",\n    "mean_rmg_delivery_service_min",\n])]\nprint(state_ablation.to_string(index=False))\n')

## 4. Rerun the saved baseline and what-if models

The default run uses 10 matched seeds, 3 saved models, and 17,892 cases per model. It can take about 30 minutes. For a short mechanics test, change `RUN_FULL_SCENARIOS = True` to `False` in the code string below.

In [ ]:
run_in_reproduction_environment('\nimport reviewer_runner as rr\n\nRUN_FULL_SCENARIOS = True\nmode = "full" if RUN_FULL_SCENARIOS else "smoke"\n\noutput_dir = rr.run_saved_models(mode=mode)\nprint(f"Fresh scenario outputs: {output_dir}")\n\nif RUN_FULL_SCENARIOS:\n    comparison = rr.compare_with_frozen_results(output_dir)\n    comparison.to_csv(\n        output_dir / "comparison_with_thesis_results.csv", index=False\n    )\n    print(comparison.to_string(index=False))\n    assert comparison["values_match"].all()\n    print("Full reproduction passed: all scenario tables match the thesis results.")\nelse:\n    print("Smoke test passed. It verifies execution but not the thesis values.")\n')

## Interpretation

The rerun establishes that the saved model and its two interventions are executable and reproducible. It does not establish physical causal effects at the terminal. The model does not contain explicit container locations, crane trajectories, physical transit, or queue states.